# 📊 Mini Project 3B — Pekan 4: Full Experiment & Statistical Analysis
**Gas Optimization Framework untuk Smart Contract Solidity**

Jalankan sel dari atas ke bawah secara berurutan.

| Sel | Isi |
|---|---|
| 1 | Setup & load semua hasil dari Pekan 2 & 3 |
| 2 | Tabel 4.4a — Gas savings per anti-pattern |
| 3 | Tabel 4.4b & 4.4c — Domain distribution & Top 10 contracts |
| 4 | Tabel 4.4d — Complexity analysis + korelasi Spearman |
| 5 | Uji Wilcoxon signed-rank (H1: gas savings signifikan) |
| 6 | Tabel 4.4e — Perbandingan detektor kita vs Slither |
| 7 | Uji Chi-square & Kruskal-Wallis + ringkasan statistik |
| 8 | Export CSV + Checklist akhir Pekan 4 |

---
## ⚙️ Sel 1 — Setup & Load Data

In [1]:
import sys, json, csv
import numpy as np
from pathlib import Path
from collections import defaultdict

ROOT        = Path().resolve().parent
RESULTS_DIR = ROOT / 'results_50'
RESULTS_DIR.mkdir(exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Install scipy jika belum ada
try:
    from scipy import stats
    print('✅ scipy tersedia')
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'scipy', '--quiet'], check=True)
    from scipy import stats
    print('✅ scipy berhasil diinstall')

from src.detectors import ALL_DETECTORS
DETECTOR_NAMES = [n for n, _ in ALL_DETECTORS]

# ── Load semua hasil
with open(RESULTS_DIR / 'pekan2_detector_results.json') as f:
    p2_results = json.load(f)

with open(RESULTS_DIR / 'pekan3_gas_benchmark.json') as f:
    bench_results = json.load(f)

try:
    with open(RESULTS_DIR / 'pekan3_slither_results.json') as f:
        slither_results = json.load(f)
    print(f'✅ Slither results: {len(slither_results)} kontrak')
except FileNotFoundError:
    slither_results = {}
    print('⚠️  Slither results tidak ditemukan — Sel 6 akan menampilkan data kosong')

compiled   = [r for r in p2_results if r['compile_ok']]
p2_by_name = {r['nama']: r for r in compiled}

total_findings = sum(sum(r.get(d, 0) for d in DETECTOR_NAMES) for r in compiled)

print(f'\n✅ Setup selesai')
print(f'   Dataset total      : {len(p2_results)} kontrak')
print(f'   Berhasil compile   : {len(compiled)} kontrak')
print(f'   Benchmark patterns : {len(bench_results)}/6')
print(f'   Total findings     : {total_findings:,}')

✅ scipy tersedia
✅ Slither results: 10 kontrak

✅ Setup selesai
   Dataset total      : 50 kontrak
   Berhasil compile   : 50 kontrak
   Benchmark patterns : 6/6
   Total findings     : 1,383


---
## ⛽ Sel 2 — Tabel 4.4a: Gas Savings per Anti-Pattern

In [2]:
print('=== TABEL 4.4a: GAS SAVINGS PER ANTI-PATTERN ===\n')
print(f'{"No":>3}  {"Anti-Pattern":<28} {"Gas Boros":>10} {"Gas Hemat":>10} '
      f'{"Selisih":>9} {"Hemat (%)": >10}')
print('─' * 75)
for i, r in enumerate(bench_results, 1):
    print(f'{i:>3}. {r["pattern"]:<28} '
          f'{r["boros_gas"]:>10,} '
          f'{r["hemat_gas"]:>10,} '
          f'{r["diff_gas"]:>9,} '
          f'{r["pct_save"]:>9.2f}%')
print('─' * 75)

positive = [r for r in bench_results if r['diff_gas'] > 0]
avg_pct  = sum(r['pct_save'] for r in bench_results) / len(bench_results)
max_r    = max(bench_results, key=lambda x: x['pct_save'])
print(f'\nPattern dengan penghematan positif : {len(positive)}/6')
print(f'Rata-rata penghematan (6 pattern)  : {avg_pct:.2f}%')
print(f'Penghematan terbesar               : {max_r["pattern"]} ({max_r["pct_save"]:.2f}%)')
print()
print('Keterangan:')
print('  dead_code = 0%: compiler mengelimasi internal dead functions bahkan')
print('  tanpa optimizer, sehingga deployment bytecode identik.')

csv_4a = RESULTS_DIR / 'tabel_4_4a_gas_savings.csv'
with open(csv_4a, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['pattern','label','boros_gas','hemat_gas','diff_gas','pct_save'])
    w.writeheader()
    w.writerows(bench_results)
print(f'\n💾 {csv_4a}')

=== TABEL 4.4a: GAS SAVINGS PER ANTI-PATTERN ===

 No  Anti-Pattern                  Gas Boros  Gas Hemat   Selisih  Hemat (%)
───────────────────────────────────────────────────────────────────────────
  1. redundant_sload                  24,208     24,022       186      0.77%
  2. unoptimized_loop                 51,187     50,156     1,031      2.01%
  3. string_vs_bytes32                24,540     23,590       950      3.87%
  4. public_vs_external               52,544     49,871     2,673      5.09%
  5. unchecked_arithmetic             59,105     47,060    12,045     20.38%
  6. dead_code                       123,985    123,985         0      0.00%
───────────────────────────────────────────────────────────────────────────

Pattern dengan penghematan positif : 5/6
Rata-rata penghematan (6 pattern)  : 5.35%
Penghematan terbesar               : unchecked_arithmetic (20.38%)

Keterangan:
  dead_code = 0%: compiler mengelimasi internal dead functions bahkan
  tanpa optimizer, sehin

---
## 📂 Sel 3 — Tabel 4.4b & 4.4c: Domain Distribution & Top 10

In [3]:
# ── Tabel 4.4b: distribusi per domain
domains = ['DeFi', 'NFT', 'Token', 'Governance', 'Utility']

print('=== TABEL 4.4b: DISTRIBUSI FINDINGS PER DOMAIN ===\n')
header = f'{"Domain":<14}' + ''.join(f'{n[:7]:>9}' for n in DETECTOR_NAMES) + f'{"TOTAL":>8} {"n":>4}'
print(header)
print('─' * (14 + 9*len(DETECTOR_NAMES) + 12))

domain_rows = []
for domain in domains:
    dr   = [r for r in compiled if r['domain'] == domain]
    vals = [sum(r.get(d, 0) for r in dr) for d in DETECTOR_NAMES]
    row  = {'domain': domain, 'n': len(dr), 'total': sum(vals)}
    row.update(dict(zip(DETECTOR_NAMES, vals)))
    domain_rows.append(row)
    print(f'{domain:<14}' + ''.join(f'{v:>9}' for v in vals) + f'{sum(vals):>8} {len(dr):>4}')

print('─' * (14 + 9*len(DETECTOR_NAMES) + 12))
totals_per_det = [sum(r.get(d, 0) for r in compiled) for d in DETECTOR_NAMES]
print(f'{"TOTAL":<14}' + ''.join(f'{v:>9}' for v in totals_per_det)
      + f'{sum(totals_per_det):>8} {len(compiled):>4}')

# ── Tabel 4.4c: top 10 contracts
print('\n=== TABEL 4.4c: TOP 10 KONTRAK DENGAN FINDINGS TERBANYAK ===\n')
print(f'{"No":>3}  {"Nama":<35} {"Domain":<12} {"LOC":>6} {"Total":>7}')
print('─' * 70)

top10 = sorted(compiled,
               key=lambda r: sum(r.get(d, 0) for d in DETECTOR_NAMES),
               reverse=True)[:10]
for i, r in enumerate(top10, 1):
    total = sum(r.get(d, 0) for d in DETECTOR_NAMES)
    print(f'{i:>3}. {r["nama"]:<35} {r["domain"]:<12} {r["loc"]:>6,} {total:>7}')

# Save CSVs
csv_4b = RESULTS_DIR / 'tabel_4_4b_domain.csv'
with open(csv_4b, 'w', newline='') as f:
    fields = ['domain','n','total'] + DETECTOR_NAMES
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader(); w.writerows(domain_rows)

csv_4c = RESULTS_DIR / 'tabel_4_4c_top10.csv'
with open(csv_4c, 'w', newline='') as f:
    rows = [{'nama': r['nama'], 'domain': r['domain'], 'loc': r['loc'],
              'total': sum(r.get(d,0) for d in DETECTOR_NAMES),
              **{d: r.get(d, 0) for d in DETECTOR_NAMES}} for r in top10]
    w = csv.DictWriter(f, fieldnames=['nama','domain','loc','total'] + DETECTOR_NAMES)
    w.writeheader(); w.writerows(rows)

print(f'\n💾 {csv_4b}')
print(f'💾 {csv_4c}')

=== TABEL 4.4b: DISTRIBUSI FINDINGS PER DOMAIN ===

Domain          redunda  unoptim  string_  public_  uncheck  dead_co   TOTAL    n
────────────────────────────────────────────────────────────────────────────────
DeFi                148        2       22       45        0       23     240   10
NFT                 208        0       94      150        0       34     486   10
Token               110        0       27      179        0       27     343   10
Governance           67        0       52       80        0       16     215   10
Utility              28        5        2       64        0        0      99   10
────────────────────────────────────────────────────────────────────────────────
TOTAL               561        7      197      518        0      100    1383   50

=== TABEL 4.4c: TOP 10 KONTRAK DENGAN FINDINGS TERBANYAK ===

 No  Nama                                Domain          LOC   Total
──────────────────────────────────────────────────────────────────────
  1. DCLR

---
## 📐 Sel 4 — Tabel 4.4d: Complexity Analysis + Korelasi Spearman

In [4]:
from scipy.stats import spearmanr

# ── Tabel 4.4d
print('=== TABEL 4.4d: ANALISIS PER COMPLEXITY LEVEL ===\n')
print(f'{"Complexity":<12} {"n":>4} {"Avg LOC":>9} {"Avg Findings":>14} {"Density (%)": >12}')
print('─' * 57)

complexity_rows = []
for cx in ['Simple', 'Medium', 'Complex']:
    group = [r for r in compiled if r.get('complexity') == cx]
    if not group:
        print(f'{cx:<12} {0:>4}  (tidak ada kontrak di level ini)')
        continue
    avg_loc  = sum(r['loc'] for r in group) / len(group)
    avg_find = sum(sum(r.get(d, 0) for d in DETECTOR_NAMES) for r in group) / len(group)
    density  = avg_find / avg_loc * 100 if avg_loc else 0
    complexity_rows.append({'complexity': cx, 'n': len(group),
                             'avg_loc': round(avg_loc,1), 'avg_findings': round(avg_find,2),
                             'density_pct': round(density,4)})
    print(f'{cx:<12} {len(group):>4} {avg_loc:>9.0f} {avg_find:>14.1f} {density:>11.3f}%')

# ── Korelasi Spearman: LOC vs total findings
locs   = [r['loc'] for r in compiled]
totals = [sum(r.get(d, 0) for d in DETECTOR_NAMES) for r in compiled]
rho, p_rho = spearmanr(locs, totals)

print(f'\n─── Korelasi Spearman: LOC vs Total Findings ───')
print(f'  rho = {rho:.4f}')
print(f'  p   = {p_rho:.4f}')
print(f'  Interpretasi: {"korelasi signifikan (p < 0.05)" if p_rho < 0.05 else "tidak signifikan (p ≥ 0.05)"}')
print(f'  Arah: {"positif — kontrak lebih panjang cenderung lebih banyak findings" if rho > 0 else "negatif"}')

csv_4d = RESULTS_DIR / 'tabel_4_4d_complexity.csv'
with open(csv_4d, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['complexity','n','avg_loc','avg_findings','density_pct'])
    w.writeheader(); w.writerows(complexity_rows)
print(f'\n💾 {csv_4d}')

=== TABEL 4.4d: ANALISIS PER COMPLEXITY LEVEL ===



Complexity      n   Avg LOC   Avg Findings  Density (%)
─────────────────────────────────────────────────────────
Simple          2        56            9.0      15.929%
Medium         25       293           22.0       7.515%
Complex        23      1352           35.4       2.622%

─── Korelasi Spearman: LOC vs Total Findings ───
  rho = 0.3289
  p   = 0.0197
  Interpretasi: korelasi signifikan (p < 0.05)
  Arah: positif — kontrak lebih panjang cenderung lebih banyak findings

💾 C:\Users\Ridho\project\KBJ\gas_optimization\results_50\tabel_4_4d_complexity.csv


---
## 🧪 Sel 5 — Uji Wilcoxon Signed-Rank (H1: Gas Savings Signifikan)

In [5]:
from scipy.stats import wilcoxon

print('=== UJI WILCOXON SIGNED-RANK ===')
print('H0 : Median penghematan gas = 0 (tidak ada efek optimasi)')
print('H1 : Median penghematan gas > 0 (optimasi memberi efek positif)')
print('n  : 6 pasang pengukuran (satu per anti-pattern)\n')

boros_arr = np.array([r['boros_gas'] for r in bench_results], dtype=float)
hemat_arr = np.array([r['hemat_gas'] for r in bench_results], dtype=float)
diffs     = boros_arr - hemat_arr

print(f'{"Pattern":<28} {"Boros":>9} {"Hemat":>9} {"Diff":>8} {"Rank":>6}')
print('─' * 65)
nonzero  = [(i, d) for i, d in enumerate(diffs) if d != 0]
sorted_d = sorted(nonzero, key=lambda x: abs(x[1]))
ranks    = {i: rk+1 for rk, (i, _) in enumerate(sorted_d)}
for i, r in enumerate(bench_results):
    rk = f'{ranks[i]}' if i in ranks else '-'
    print(f'{r["pattern"]:<28} {r["boros_gas"]:>9,} {r["hemat_gas"]:>9,} '
          f'{int(diffs[i]):>8,} {rk:>6}')

W_stat, p_wilcox = np.nan, np.nan
try:
    W_stat, p_wilcox = wilcoxon(diffs, alternative='greater', zero_method='wilcox')
    print(f'\nWilcoxon W+ = {W_stat:.1f}')
    print(f'p-value     = {p_wilcox:.4f}')
except Exception as e:
    # Fallback: sign test jika n terlalu kecil
    print(f'\n(wilcoxon gagal: {e} — fallback ke sign test)')
    n_pos = int(sum(d > 0 for d in diffs))
    n_neg = int(sum(d < 0 for d in diffs))
    from scipy.stats import binomtest
    bt = binomtest(n_pos, n_pos + n_neg, p=0.5, alternative='greater')
    W_stat, p_wilcox = float(n_pos), float(bt.pvalue)
    print(f'Sign test: {n_pos}+ / {n_neg}- dari {n_pos+n_neg} non-zero pairs')
    print(f'p-value   = {p_wilcox:.4f}')

alpha = 0.05
print(f'\nKeputusan (α={alpha}): '
      f'{"✅ H1 DITERIMA — penghematan gas signifikan" if p_wilcox < alpha else "❌ H1 ditolak — tidak signifikan"}')
print()
print('Catatan metodologi:')
print('  - Uji non-parametrik dipilih karena distribusi gas tidak normal')
print('  - zero_method="wilcox": pasangan dengan diff=0 (dead_code) dikeluarkan')
print('  - Uji one-sided karena hipotesis searah (penghematan > 0)')

=== UJI WILCOXON SIGNED-RANK ===
H0 : Median penghematan gas = 0 (tidak ada efek optimasi)
H1 : Median penghematan gas > 0 (optimasi memberi efek positif)
n  : 6 pasang pengukuran (satu per anti-pattern)

Pattern                          Boros     Hemat     Diff   Rank
─────────────────────────────────────────────────────────────────
redundant_sload                 24,208    24,022      186      1
unoptimized_loop                51,187    50,156    1,031      3
string_vs_bytes32               24,540    23,590      950      2
public_vs_external              52,544    49,871    2,673      4
unchecked_arithmetic            59,105    47,060   12,045      5
dead_code                      123,985   123,985        0      -

Wilcoxon W+ = 15.0
p-value     = 0.0312

Keputusan (α=0.05): ✅ H1 DITERIMA — penghematan gas signifikan

Catatan metodologi:
  - Uji non-parametrik dipilih karena distribusi gas tidak normal
  - zero_method="wilcox": pasangan dengan diff=0 (dead_code) dikeluarkan
  - Uji o

---
## 🔍 Sel 6 — Tabel 4.4e: Perbandingan Detektor Kita vs Slither

In [6]:
print('=== TABEL 4.4e: PERBANDINGAN DETEKTOR KITA vs SLITHER ===\n')

rows_4e    = []
a = b = c = d_cell = 0  # McNemar: both, only_ours, only_slither, neither

if not slither_results:
    print('⚠️  Slither results kosong — perbandingan tidak dapat dilakukan.')
    print('   Kemungkinan penyebab:')
    print('   1. Slither tidak kompatibel dengan solc versi 0.4.x (mayoritas dataset)')
    print('   2. Kontrak menggunakan pragma versi lama yang tidak didukung Slither')
    print()
    print('   Distribusi findings detektor kita pada 10 kontrak sampel:')
    sample10 = compiled[:10]
    print(f'   {"Kontrak":<35} {"Total":>7}')
    print('   ' + '─' * 45)
    for r in sample10:
        total = sum(r.get(dn, 0) for dn in DETECTOR_NAMES)
        print(f'   {r["nama"]:<35} {total:>7}')
else:
    GAS_DETECTORS = {'costly-loop', 'dead-code', 'unused-return',
                     'cache-array-length', 'storage-array', 'redundant-statements'}

    print(f'{"Kontrak":<32} {"Kita":>7} {"Slither (gas)":>13}  Status')
    print('─' * 65)
    for nama, sdata in slither_results.items():
        our_total  = sum(p2_by_name.get(nama, {}).get(dn, 0) for dn in DETECTOR_NAMES)
        slith_n    = len([x for x in sdata.get('all_detectors', []) if x in GAS_DETECTORS])
        both       = int(our_total > 0 and slith_n > 0)
        only_ours  = int(our_total > 0 and slith_n == 0)
        only_slith = int(our_total == 0 and slith_n > 0)
        neither    = int(our_total == 0 and slith_n == 0)
        status = ('Keduanya' if both else
                  'Hanya kita' if only_ours else
                  'Hanya Slither' if only_slith else 'Nihil')
        rows_4e.append({'nama': nama, 'kita': our_total, 'slither': slith_n,
                        'both': both, 'only_ours': only_ours,
                        'only_slither': only_slith, 'neither': neither})
        print(f'{nama:<32} {our_total:>7} {slith_n:>13}  {status}')
        a += both; b += only_ours; c += only_slith; d_cell += neither

    print(f'\nKontingen McNemar 2x2 (n={len(rows_4e)} kontrak):')
    print(f'                    Slither+   Slither-')
    print(f'  Detektor kita +     {a:>5}      {b:>5}   (b)')
    print(f'  Detektor kita -     {c:>5}      {d_cell:>5}')

    if b + c > 0:
        # McNemar exact (binomial)
        from scipy.stats import binomtest
        mc_res   = binomtest(min(b, c), b + c, p=0.5, alternative='two-sided')
        p_mcnemar = mc_res.pvalue
        print(f'\nMcNemar (exact binomial): p = {p_mcnemar:.4f}')
        print(f'Interpretasi: '
              f'{"perbedaan signifikan" if p_mcnemar < 0.05 else "tidak signifikan — kedua tool sebanding"}')
    else:
        p_mcnemar = np.nan
        print('\nMcNemar tidak dapat dihitung: b+c = 0')
        print('Artinya: tidak ada perbedaan antara kedua tool pada sampel ini.')
        if b == 0 and c == 0 and a == 0:
            print('Kemungkinan: Slither gagal menganalisis semua kontrak (versi solc lama).')

    if rows_4e:
        csv_4e = RESULTS_DIR / 'tabel_4_4e_slither.csv'
        with open(csv_4e, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=list(rows_4e[0].keys()))
            w.writeheader(); w.writerows(rows_4e)
        print(f'\n💾 {csv_4e}')

=== TABEL 4.4e: PERBANDINGAN DETEKTOR KITA vs SLITHER ===

Kontrak                             Kita Slither (gas)  Status
─────────────────────────────────────────────────────────────────
AppProxyUpgradeable                    0             0  Nihil
Dai                                   10             0  Hanya kita
DaiJoin                                0             0  Nihil
ETHJoin                                0             0  Nihil
GemJoin                                0             0  Nihil
KyberNetworkProxy                     68             0  Hanya kita
Spotter                                0             0  Nihil
UniswapV2Factory                      20             0  Hanya kita
UniswapV2Pair                         14             0  Hanya kita
Vat                                   24             0  Hanya kita

Kontingen McNemar 2x2 (n=10 kontrak):
                    Slither+   Slither-
  Detektor kita +         0          5   (b)
  Detektor kita -         0          5

McN

---
## 📈 Sel 7 — Uji Chi-Square, Kruskal-Wallis & Ringkasan Statistik

In [7]:
from scipy.stats import chi2_contingency, kruskal

chi2_stat = p_chi = H_stat = p_kw = np.nan

# ── Chi-square: domain vs keberadaan findings
print('=== UJI CHI-SQUARE: DOMAIN vs KEBERADAAN FINDINGS ===')
print('H0: Distribusi findings tidak bergantung pada domain\n')

contingency = []
for domain in domains:
    group = [r for r in compiled if r['domain'] == domain]
    has   = sum(1 for r in group if sum(r.get(d,0) for d in DETECTOR_NAMES) > 0)
    no    = len(group) - has
    contingency.append([has, no])
    print(f'  {domain:<12}: {has:>2} ada findings, {no:>2} tidak ada')

try:
    chi2_stat, p_chi, dof, _ = chi2_contingency(contingency)
    print(f'\nChi-square = {chi2_stat:.4f}, df = {dof}, p = {p_chi:.4f}')
    print(f'Keputusan (α=0.05): '
          f'{"H0 DITOLAK — domain berpengaruh signifikan" if p_chi < 0.05 else "H0 diterima — domain tidak berpengaruh signifikan"}')
except Exception as e:
    print(f'Chi-square gagal: {e}')

# ── Kruskal-Wallis: complexity vs jumlah findings
print('\n=== UJI KRUSKAL-WALLIS: COMPLEXITY vs JUMLAH FINDINGS ===')
print('H0: Distribusi findings sama di semua complexity level\n')

kw_groups = {}
for cx in ['Simple', 'Medium', 'Complex']:
    vals = [sum(r.get(d,0) for d in DETECTOR_NAMES)
            for r in compiled if r.get('complexity') == cx]
    if vals:
        kw_groups[cx] = vals

for cx, vals in kw_groups.items():
    print(f'  {cx:<10}: n={len(vals):>2}, median={np.median(vals):.1f}, mean={np.mean(vals):.1f}')

if len(kw_groups) >= 2:
    try:
        H_stat, p_kw = kruskal(*kw_groups.values())
        print(f'\nKruskal-Wallis H = {H_stat:.4f}, p = {p_kw:.4f}')
        print(f'Keputusan (α=0.05): '
              f'{"H0 DITOLAK — complexity berpengaruh signifikan" if p_kw < 0.05 else "H0 diterima — complexity tidak berpengaruh signifikan"}')
    except Exception as e:
        print(f'Kruskal-Wallis gagal: {e}')
else:
    print('  (kurang dari 2 grup — uji tidak dapat dilakukan)')

# ── Ringkasan statistik lengkap
print('\n=== RINGKASAN STATISTIK LENGKAP ===')
all_totals = [sum(r.get(d,0) for d in DETECTOR_NAMES) for r in compiled]
has_finding = sum(1 for t in all_totals if t > 0)
print(f'Kontrak valid          : {len(compiled)}')
print(f'Kontrak dgn findings   : {has_finding} ({has_finding/len(compiled)*100:.1f}%)')
print(f'Total findings         : {sum(all_totals):,}')
print(f'Rata-rata per kontrak  : {np.mean(all_totals):.1f}')
print(f'Median per kontrak     : {np.median(all_totals):.1f}')
print(f'Std deviation          : {np.std(all_totals):.1f}')
print(f'Min / Max              : {min(all_totals)} / {max(all_totals)}')
print()
print(f'{"Detector":<30} {"Total":>7} {"Proporsi":>9}')
print('─' * 50)
grand_total = sum(all_totals)
for name in DETECTOR_NAMES:
    t   = sum(r.get(name, 0) for r in compiled)
    pct = t / grand_total * 100 if grand_total else 0
    print(f'  {name:<28}: {t:>7,} {pct:>8.1f}%')

=== UJI CHI-SQUARE: DOMAIN vs KEBERADAAN FINDINGS ===
H0: Distribusi findings tidak bergantung pada domain

  DeFi        :  8 ada findings,  2 tidak ada
  NFT         : 10 ada findings,  0 tidak ada
  Token       :  9 ada findings,  1 tidak ada
  Governance  : 10 ada findings,  0 tidak ada
  Utility     :  8 ada findings,  2 tidak ada

Chi-square = 4.4444, df = 4, p = 0.3492
Keputusan (α=0.05): H0 diterima — domain tidak berpengaruh signifikan

=== UJI KRUSKAL-WALLIS: COMPLEXITY vs JUMLAH FINDINGS ===
H0: Distribusi findings sama di semua complexity level

  Simple    : n= 2, median=9.0, mean=9.0
  Medium    : n=25, median=24.0, mean=22.0
  Complex   : n=23, median=32.0, mean=35.4

Kruskal-Wallis H = 5.6060, p = 0.0606
Keputusan (α=0.05): H0 diterima — complexity tidak berpengaruh signifikan

=== RINGKASAN STATISTIK LENGKAP ===
Kontrak valid          : 50
Kontrak dgn findings   : 45 (90.0%)
Total findings         : 1,383
Rata-rata per kontrak  : 27.7
Median per kontrak     : 26.5
Std 

---
## ✅ Sel 8 — Export CSV & Checklist Akhir Pekan 4

In [8]:
# ── Export: semua findings per kontrak × detector
all_rows = []
for r in compiled:
    for dname in DETECTOR_NAMES:
        cnt = r.get(dname, 0)
        if cnt > 0:
            all_rows.append({
                'nama': r['nama'], 'domain': r['domain'],
                'complexity': r.get('complexity', ''),
                'loc': r['loc'], 'detector': dname, 'count': cnt
            })

csv_all = RESULTS_DIR / 'pekan4_all_findings.csv'
with open(csv_all, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['nama','domain','complexity','loc','detector','count'])
    w.writeheader(); w.writerows(all_rows)

def _safe(v):
    return None if (v is None or (isinstance(v, float) and np.isnan(v))) else float(v)

stat_summary = {
    'wilcoxon_gas_savings': {
        'W': _safe(W_stat), 'p': _safe(p_wilcox),
        'significant_005': bool(not np.isnan(p_wilcox) and p_wilcox < 0.05)
    },
    'chi_square_domain': {
        'chi2': _safe(chi2_stat), 'p': _safe(p_chi),
        'significant_005': bool(not np.isnan(p_chi) and p_chi < 0.05)
    },
    'kruskal_wallis_complexity': {
        'H': _safe(H_stat), 'p': _safe(p_kw),
        'significant_005': bool(not np.isnan(p_kw) and p_kw < 0.05)
    },
    'spearman_loc_findings': {
        'rho': _safe(rho), 'p': _safe(p_rho),
        'significant_005': bool(not np.isnan(p_rho) and p_rho < 0.05)
    },
}
stat_path = RESULTS_DIR / 'pekan4_statistical_tests.json'
with open(stat_path, 'w') as f:
    json.dump(stat_summary, f, indent=2)

print('💾 File yang disimpan:')
for p in [csv_all, stat_path,
           RESULTS_DIR/'tabel_4_4a_gas_savings.csv',
           RESULTS_DIR/'tabel_4_4b_domain.csv',
           RESULTS_DIR/'tabel_4_4c_top10.csv',
           RESULTS_DIR/'tabel_4_4d_complexity.csv']:
    exists = '✅' if Path(p).exists() else '❌'
    print(f'  {exists} {Path(p).name}')

# ── Checklist akhir Pekan 4
print('\n' + '='*55)
print('  CHECKLIST AKHIR PEKAN 4')
print('='*55)

n_total   = len(p2_results)
n_compile = len(compiled)

checks = [
    (f'Full run {n_total} kontrak selesai',
        n_total > 0),
    (f'{n_compile} kontrak berhasil compile',
        n_compile >= 40),
    ('Tabel 4.4a (gas savings) tersedia',
        (RESULTS_DIR / 'tabel_4_4a_gas_savings.csv').exists()),
    ('Tabel 4.4b (domain dist.) tersedia',
        (RESULTS_DIR / 'tabel_4_4b_domain.csv').exists()),
    ('Tabel 4.4c (top 10) tersedia',
        (RESULTS_DIR / 'tabel_4_4c_top10.csv').exists()),
    ('Tabel 4.4d (complexity) tersedia',
        (RESULTS_DIR / 'tabel_4_4d_complexity.csv').exists()),
    ('Uji Wilcoxon selesai',
        not np.isnan(p_wilcox)),
    ('Uji Chi-square selesai',
        not np.isnan(chi2_stat)),
    ('Uji Kruskal-Wallis selesai',
        not np.isnan(H_stat)),
    ('Korelasi Spearman selesai',
        not np.isnan(rho)),
    ('CSV semua findings tersedia',
        csv_all.exists()),
    ('Ringkasan uji statistik tersimpan',
        stat_path.exists()),
]

semua_ok = True
for label, ok in checks:
    icon = '✅' if ok else '❌'
    if not ok: semua_ok = False
    print(f'  {icon} {label}')

print('─'*55)
print(f'  Total findings         : {sum(all_totals):,}')
print(f'  Avg gas savings        : {sum(r["pct_save"] for r in bench_results)/len(bench_results):.1f}%')
p_disp = f'{p_wilcox:.5f}' if not np.isnan(p_wilcox) else 'N/A'
sig_str = ('signifikan ✅' if not np.isnan(p_wilcox) and p_wilcox < 0.05
           else 'tidak signifikan' if not np.isnan(p_wilcox) else 'N/A')
print(f'  Wilcoxon p-value       : {p_disp} ({sig_str})')
print(f'  Slither comparison     : {len(slither_results)} kontrak')
print('='*55)
print('  🎉 Eksperimen selesai! Siap laporan akhir.' if semua_ok
      else '  ⚠️  Cek item ❌ di atas')
print('='*55)

💾 File yang disimpan:
  ✅ pekan4_all_findings.csv
  ✅ pekan4_statistical_tests.json
  ✅ tabel_4_4a_gas_savings.csv
  ✅ tabel_4_4b_domain.csv
  ✅ tabel_4_4c_top10.csv
  ✅ tabel_4_4d_complexity.csv

  CHECKLIST AKHIR PEKAN 4
  ✅ Full run 50 kontrak selesai
  ✅ 50 kontrak berhasil compile
  ✅ Tabel 4.4a (gas savings) tersedia
  ✅ Tabel 4.4b (domain dist.) tersedia
  ✅ Tabel 4.4c (top 10) tersedia
  ✅ Tabel 4.4d (complexity) tersedia
  ✅ Uji Wilcoxon selesai
  ✅ Uji Chi-square selesai
  ✅ Uji Kruskal-Wallis selesai
  ✅ Korelasi Spearman selesai
  ✅ CSV semua findings tersedia
  ✅ Ringkasan uji statistik tersimpan
───────────────────────────────────────────────────────
  Total findings         : 1,383
  Avg gas savings        : 5.4%
  Wilcoxon p-value       : 0.03125 (signifikan ✅)
  Slither comparison     : 10 kontrak
  🎉 Eksperimen selesai! Siap laporan akhir.
